**Environment Setup**  
بررسی محیط پایتون و مسیر اجرایی

In [ ]:
import sys; print(sys.executable)

**Embedding & Reranker Models**  
بارگذاری و آزمایش مدل‌های Embedding و Reranker به‌صورت آفلاین

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"       
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from sentence_transformers import SentenceTransformer, CrossEncoder

EMBED_PATH    = "models/multilingual-e5-small"
RERANKER_PATH = "models/mmarco-reranker"

# --- تست embedding ---
embedder = SentenceTransformer(EMBED_PATH, device="cpu")
vec = embedder.encode("type 2 diabetes symptoms")
print("embedding OK | dim =", len(vec))

# --- تست reranker ---
reranker = CrossEncoder(RERANKER_PATH, device="cpu")
score = reranker.predict([("what is diabetes?", "Diabetes is a metabolic disease.")])
print("reranker OK | score =", score)

**Raw Data Creation**  
استخراج متن از فایل آموزشی FAQ متنی و صفحه وب HTML

In [ ]:
with open("data/raw/faq.txt", "w", encoding="utf-8") as f:
    f.write(faq_text)

with open("data/raw/web_article.html", "w", encoding="utf-8") as f:
    f.write(html_text)

print("created:", os.listdir("data/raw"))

**PDF Inspection**  
بررسی و استخراج متن از فایل PDF آموزشی دیابت

In [ ]:
from pypdf import PdfReader

reader = PdfReader("data/raw/diabetes_guide.pdf")
print("تعداد صفحات:", len(reader.pages))

# نمونه متن از چند صفحه اول
for i in range(min(3, len(reader.pages))):
    text = reader.pages[i].extract_text() or ""
    print(f"\n--- صفحه {i+1} | طول متن: {len(text)} کاراکتر ---")
    print(text[:300])

**Document Ingestion**  
تابع بارگذاری یکپارچه اسناد از فرمت‌های PDF، HTML و TXT

In [ ]:
import os, glob
from pypdf import PdfReader
from bs4 import BeautifulSoup

def load_pdf(path):
    reader = PdfReader(path)
    pages = [(p.extract_text() or "") for p in reader.pages]
    return "\n".join(pages)

def load_html(path):
    with open(path, encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    # حذف تگ‌های اسکریپت/استایل و گرفتن متن تمیز
    for tag in soup(["script", "style"]):
        tag.decompose()
    return soup.get_text(separator="\n", strip=True)

def load_txt(path):
    with open(path, encoding="utf-8") as f:
        return f.read()

def ingest_raw(raw_dir="data/raw"):
    docs = []
    loaders = {".pdf": ("pdf", load_pdf),
               ".html": ("web", load_html),
               ".txt": ("faq", load_txt)}
    for path in sorted(glob.glob(os.path.join(raw_dir, "*"))):
        ext = os.path.splitext(path)[1].lower()
        if ext not in loaders:
            continue
        source_type, loader = loaders[ext]
        text = loader(path).strip()
        docs.append({
            "text": text,
            "metadata": {
                "source": os.path.basename(path),
                "source_type": source_type,
            }
        })
    return docs

documents = ingest_raw()
for d in documents:
    print(f"{d['metadata']['source']:25} | type={d['metadata']['source_type']:4} | chars={len(d['text'])}")

**Recursive Chunking**  
چانک‌بندی بازگشتی با دو اندازه کوچک (۳۰۰) و بزرگ (۶۰۰) و مقایسه نتایج

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_recursive(docs, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", "؟", ".", "،", " ", ""],  # جداکننده‌های مناسب فارسی
        length_function=len,
    )
    chunks = []
    for doc in docs:
        for i, piece in enumerate(splitter.split_text(doc["text"])):
            chunks.append({
                "text": piece,
                "metadata": {
                    **doc["metadata"],
                    "chunk_method": "recursive",
                    "chunk_size": chunk_size,
                    "chunk_id": f"{doc['metadata']['source']}_rec{chunk_size}_{i}",
                }
            })
    return chunks

# دو اندازه: کوچک و بزرگ
chunks_rec_small = chunk_recursive(documents, chunk_size=300, chunk_overlap=50)
chunks_rec_large = chunk_recursive(documents, chunk_size=600, chunk_overlap=100)

print(f"recursive small (300): {len(chunks_rec_small)} chunks")
print(f"recursive large (600): {len(chunks_rec_large)} chunks")
print("\n--- نمونه chunk کوچک ---")
print(chunks_rec_small[0]["text"])

**Semantic Chunking**  
چانک‌بندی معنایی بر اساس شباهت کسینوسی بین جملات متوالی

In [ ]:
import re
import numpy as np

def split_sentences_fa(text):
    # شکستن به جمله بر اساس نقطه، علامت سوال و خط جدید
    parts = re.split(r"(?<=[.؟!])\s+|\n+", text)
    return [s.strip() for s in parts if len(s.strip()) > 10]

def chunk_semantic(docs, threshold=0.65, max_chars=600):
    chunks = []
    for doc in docs:
        sents = split_sentences_fa(doc["text"])
        if len(sents) < 2:
            continue
        embs = embedder.encode(sents, normalize_embeddings=True)
        # شباهت کسینوسی بین جملات متوالی (چون نرمال‌اند، ضرب داخلی = کسینوس)
        current, buf = [], ""
        for i, sent in enumerate(sents):
            if i > 0:
                sim = float(np.dot(embs[i], embs[i-1]))
                # برش اگر معنا عوض شد یا طول از حد گذشت
                if sim < threshold or len(buf) + len(sent) > max_chars:
                    chunks.append((buf, doc["metadata"]))
                    buf = ""
            buf = (buf + " " + sent).strip()
        if buf:
            chunks.append((buf, doc["metadata"]))

    out = []
    for i, (txt, meta) in enumerate(chunks):
        out.append({"text": txt, "metadata": {
            **meta, "chunk_method": "semantic", "chunk_size": "auto",
            "chunk_id": f"{meta['source']}_sem_{i}"}})
    return out

chunks_semantic = chunk_semantic(documents)
print(f"semantic: {len(chunks_semantic)} chunks")
print("\n--- نمونه chunk معنایی ---")
print(chunks_semantic[0]["text"][:400])

**Chunk Comparison**  
مقایسه آماری روش‌های چانک‌بندی از نظر تعداد، میانگین و انحراف معیار طول

In [ ]:
import numpy as np

def chunk_stats(name, chunks):
    lengths = [len(c["text"]) for c in chunks]
    return {
        "روش": name,
        "تعداد": len(chunks),
        "میانگین طول": round(np.mean(lengths)),
        "کمینه": min(lengths),
        "بیشینه": max(lengths),
        "انحراف معیار": round(np.std(lengths)),
    }

import pandas as pd
comparison = pd.DataFrame([
    chunk_stats("recursive-300", chunks_rec_small),
    chunk_stats("recursive-600", chunks_rec_large),
    chunk_stats("semantic",      chunks_semantic),
])
print(comparison.to_string(index=False))

## جمع‌بندی فاز ۲ — چانک‌بندی

سه روش پیاده و مقایسه شد:
- **Recursive-300**: بیشترین تعداد chunk، دقت بالا اما زمینه کم.
- **Recursive-600**: تعادل بین دقت و زمینه — **به‌عنوان chunk فعال انتخاب شد**.
- **Semantic**: مرزبندی بر اساس معنا، طول متغیر؛ برای متون با موضوعات متنوع مفید.

دلیل انتخاب Recursive-600: متون آموزشی ما پاراگراف‌محورند و این اندازه، پاسخ‌های پزشکی را همراه با زمینه کافی نگه می‌دارد بدون اینکه نویز اضافه وارد بازیابی کند.

**PDF Re-inspection**  
بررسی مجدد محتوای فایل PDF پیش از ادامه پایپ‌لاین

In [ ]:
from pypdf import PdfReader

reader = PdfReader("data/raw/diabetes_guide.pdf")
print("تعداد صفحات:", len(reader.pages))

# نمونه متن از چند صفحه اول
for i in range(min(3, len(reader.pages))):
    text = reader.pages[i].extract_text() or ""
    print(f"\n--- صفحه {i+1} | طول متن: {len(text)} کاراکتر ---")
    print(text[:300])

**Active Chunk Selection**  
انتخاب روش چانک‌بندی فعال (Recursive-600) برای ادامه پایپ‌لاین

In [ ]:
# انتخاب chunk فعال برای کل ادامه پایپ‌لاین
active_chunks = chunks_rec_large   # recursive-600

print(f"chunk فعال: recursive-600 | تعداد: {len(active_chunks)}")
print("نمونه:", active_chunks[0]["text"][:150])

**ChromaDB Vector Store**  
ساخت پایگاه برداری با ChromaDB و ایندکس‌گذاری چانک‌های فعال

In [ ]:
import chromadb

# کلاینت persist روی دیسک
chroma_client = chromadb.PersistentClient(path="indexes/chroma")

# اگر کالکشن قبلی هست، حذف کن تا تمیز بسازیم (در توسعه)
try:
    chroma_client.delete_collection("diabetes")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="diabetes",
    metadata={"hnsw:space": "cosine"}  # شباهت کسینوسی
)

# تابع embedding با پیشوند مناسب e5
def embed_passages(texts):
    prefixed = [f"passage: {t}" for t in texts]
    return embedder.encode(prefixed, normalize_embeddings=True).tolist()

# ساخت embedding برای همه chunkهای فعال
texts = [c["text"] for c in active_chunks]
metas = [c["metadata"] for c in active_chunks]
ids   = [c["metadata"]["chunk_id"] for c in active_chunks]
embeddings = embed_passages(texts)

collection.add(documents=texts, embeddings=embeddings, metadatas=metas, ids=ids)
print(f"در Chroma ذخیره شد: {collection.count()} بردار | بُعد: {len(embeddings[0])}")

**Dense Retrieval**  
جستجوی معنایی (متراکم) بر اساس بردار embedding کوئری

In [ ]:
def embed_query(text):
    return embedder.encode(f"query: {text}", normalize_embeddings=True).tolist()

def dense_search(query, k=5):
    q_emb = embed_query(query)
    res = collection.query(query_embeddings=[q_emb], n_results=k)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({
            "text": doc,
            "metadata": meta,
            "score": 1 - dist,           # تبدیل فاصله به شباهت
        })
    return hits

# تست با یک کوئری فارسی
query = "علائم دیابت نوع ۲ چیست؟"
results = dense_search(query, k=3)
for i, h in enumerate(results, 1):
    print(f"#{i} | score={h['score']:.3f} | {h['metadata']['source']}")
    print("   ", h["text"][:120], "\n")

**BM25 Sparse Retrieval**  
جستجوی کلیدواژه‌ای (پراکنده) با الگوریتم BM25 و توکن‌سازی فارسی

In [ ]:
import re, pickle, os
from rank_bm25 import BM25Okapi

def normalize_fa(text):
    text = text.replace("ي", "ی").replace("ك", "ک")  # یکدست‌سازی عربی→فارسی
    text = re.sub(r"[^\w\s]", " ", text)               # حذف نقطه‌گذاری
    return text

def tokenize_fa(text):
    return normalize_fa(text).split()

# توکن‌سازی همه chunkهای فعال
bm25_corpus = [tokenize_fa(c["text"]) for c in active_chunks]
bm25 = BM25Okapi(bm25_corpus)

# ذخیره روی دیسک (ایندکس + خود chunkها برای بازیابی)
os.makedirs("indexes", exist_ok=True)
with open("indexes/bm25.pkl", "wb") as f:
    pickle.dump({"bm25": bm25, "chunks": active_chunks}, f)

print(f"BM25 ساخته شد روی {len(bm25_corpus)} chunk و در indexes/bm25.pkl ذخیره شد")

# تست سریع
def bm25_search(query, k=5):
    scores = bm25.get_scores(tokenize_fa(query))
    top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [{"text": active_chunks[i]["text"],
             "metadata": active_chunks[i]["metadata"],
             "score": float(scores[i])} for i in top]

for i, h in enumerate(bm25_search("علائم دیابت", k=3), 1):
    print(f"#{i} | score={h['score']:.3f} | {h['metadata']['source']}")
    print("   ", h["text"][:100], "\n")

**Hybrid Search — RRF**  
ترکیب جستجوی متراکم و پراکنده با روش Reciprocal Rank Fusion

In [ ]:
def hybrid_search(query, k=5, rrf_k=60, pool=10):
    # گرفتن نتایج از هر دو روش (تعداد بیشتر برای ادغام بهتر)
    dense_hits = dense_search(query, k=pool)
    sparse_hits = bm25_search(query, k=pool)

    # محاسبه امتیاز RRF بر اساس رتبه در هر لیست
    scores = {}
    pool_map = {}
    for rank, h in enumerate(dense_hits):
        cid = h["metadata"]["chunk_id"]
        scores[cid] = scores.get(cid, 0) + 1 / (rrf_k + rank)
        pool_map[cid] = h
    for rank, h in enumerate(sparse_hits):
        cid = h["metadata"]["chunk_id"]
        scores[cid] = scores.get(cid, 0) + 1 / (rrf_k + rank)
        pool_map[cid] = h

    # مرتب‌سازی نهایی بر اساس امتیاز RRF
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return [{"text": pool_map[cid]["text"],
             "metadata": pool_map[cid]["metadata"],
             "score": s} for cid, s in ranked]

# تست
query = "علائم دیابت نوع ۲ چیست؟"
print("=== Hybrid (RRF) ===")
for i, h in enumerate(hybrid_search(query, k=3), 1):
    print(f"#{i} | rrf={h['score']:.4f} | {h['metadata']['source']}")
    print("   ", h["text"][:120], "\n")

**Filtered Search**  
جستجوی فیلترشده بر اساس نوع منبع (PDF، FAQ، Web)

In [ ]:
def dense_search_filtered(query, k=5, where=None):
    q_emb = embed_query(query)
    res = collection.query(query_embeddings=[q_emb], n_results=k, where=where)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"text": doc, "metadata": meta, "score": 1 - dist})
    return hits

def bm25_search_filtered(query, k=5, source_type=None):
    scores = bm25.get_scores(tokenize_fa(query))
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    hits = []
    for i in idx:
        meta = active_chunks[i]["metadata"]
        if source_type and meta["source_type"] != source_type:
            continue
        hits.append({"text": active_chunks[i]["text"], "metadata": meta, "score": float(scores[i])})
        if len(hits) >= k:
            break
    return hits

# تست: فقط از منبع نوع faq جستجو کن
print("=== فیلتر فقط FAQ (dense) ===")
for h in dense_search_filtered("علائم دیابت", k=3, where={"source_type": "faq"}):
    print(f"{h['metadata']['source']:20} | type={h['metadata']['source_type']} | score={h['score']:.3f}")

print("\n=== بدون فیلتر برای مقایسه ===")
for h in dense_search_filtered("علائم دیابت", k=3):
    print(f"{h['metadata']['source']:20} | type={h['metadata']['source_type']} | score={h['score']:.3f}")

**Reranking — CrossEncoder**  
رتبه‌بندی مجدد نتایج بازیابی با مدل CrossEncoder

In [ ]:
def rerank(query, candidates, top_k=3):
    if not candidates:
        return []
    # ساخت جفت‌های (کوئری، متن) برای cross-encoder
    pairs = [(query, c["text"]) for c in candidates]
    scores = reranker.predict(pairs)
    # چسباندن امتیاز جدید و مرتب‌سازی نزولی
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    ranked = sorted(candidates, key=lambda c: c["rerank_score"], reverse=True)
    return ranked[:top_k]

# پایپ‌لاین کامل: hybrid → rerank
def retrieve_and_rerank(query, pool=10, top_k=3, where=None):
    candidates = hybrid_search(query, k=pool)
    return rerank(query, candidates, top_k=top_k)

# تست و مقایسه ترتیب قبل/بعد از rerank
query = "علائم دیابت نوع ۲ چیست؟"
print("=== قبل از rerank (hybrid top-5) ===")
for i, h in enumerate(hybrid_search(query, k=5), 1):
    print(f"#{i} | {h['metadata']['source']} | {h['text'][:60]}")

print("\n=== بعد از rerank (top-3) ===")
for i, h in enumerate(retrieve_and_rerank(query, pool=10, top_k=3), 1):
    print(f"#{i} | rerank={h['rerank_score']:.3f} | {h['metadata']['source']} | {h['text'][:60]}")

**MMR Diversity Selection**  
انتخاب متنوع با الگوریتم Maximal Marginal Relevance برای کاهش تکرار

In [ ]:
import numpy as np

def mmr_select(query, candidates, top_k=3, lambda_param=0.7):
    if len(candidates) <= top_k:
        return candidates
    q_emb = np.array(embed_query(query))
    # بردارسازی متن کاندیداها (با پیشوند passage مثل ایندکس)
    cand_embs = embedder.encode([f"passage: {c['text']}" for c in candidates],
                                normalize_embeddings=True)
    selected, remaining = [], list(range(len(candidates)))
    while len(selected) < top_k and remaining:
        best_idx, best_score = None, -1e9
        for i in remaining:
            relevance = float(np.dot(cand_embs[i], q_emb))
            diversity = 0.0
            if selected:
                diversity = max(float(np.dot(cand_embs[i], cand_embs[j])) for j in selected)
            mmr = lambda_param * relevance - (1 - lambda_param) * diversity
            if mmr > best_score:
                best_score, best_idx = mmr, i
        selected.append(best_idx)
        remaining.remove(best_idx)
    return [candidates[i] for i in selected]

# تست: مقایسه top-3 معمولی با top-3 پس از MMR
query = "دیابت نوع ۲ چیست و چه عوارضی دارد؟"
pool = hybrid_search(query, k=10)
print("=== بدون MMR (top-3 hybrid) ===")
for i, h in enumerate(pool[:3], 1):
    print(f"#{i} | {h['text'][:70]}")

print("\n=== با MMR (top-3, lambda=0.7) ===")
for i, h in enumerate(mmr_select(query, pool, top_k=3, lambda_param=0.7), 1):
    print(f"#{i} | {h['text'][:70]}")

**LLM Client Setup**  
راه‌اندازی اتصال به مدل زبانی از طریق GapGPT و تست اولیه

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)  # خواندن .env

GAPGPT_BASE_URL = os.getenv("GAPGPT_BASE_URL", "https://api.gapgpt.app/v1")
GAPGPT_MODEL    = os.getenv("GAPGPT_MODEL", "gapgpt-qwen-3.5")
API_KEY         = os.getenv("GAPGPT_API_KEY")

assert API_KEY, "کلید API در فایل .env پیدا نشد!"

client = OpenAI(api_key=API_KEY, base_url=GAPGPT_BASE_URL)

# تست اتصال با یک پیام کوتاه
resp = client.chat.completions.create(
    model=GAPGPT_MODEL,
    messages=[{"role": "user", "content": "بگو: اتصال برقرار است"}],
    max_tokens=20,
)
print("مدل:", GAPGPT_MODEL)
print("پاسخ:", resp.choices[0].message.content)

**RAG Answer Generation**  
تولید پاسخ نهایی با پایپ‌لاین کامل RAG و Prompt سیستمی مخصوص پزشکی

In [ ]:
SYSTEM_PROMPT = """تو یک دستیار پاسخ‌گوی پزشکی درباره دیابت نوع ۲ هستی.
قوانین مهم:
۱. فقط و فقط بر اساس «زمینه» داده‌شده پاسخ بده. از دانش خودت چیزی اضافه نکن.
۲. اگر پاسخ سوال در زمینه نبود، بگو: «بر اساس منابع موجود، پاسخی برای این سوال ندارم.»
۳. در پایان هر ادعا، شماره منبع مربوطه را به شکل [۱]، [۲] و... ذکر کن.
۴. پاسخ را کوتاه، دقیق و به زبان فارسی بده.
۵. یادآوری کن که این اطلاعات جایگزین مشاوره پزشک نیست."""

def build_context(hits):
    blocks = []
    for i, h in enumerate(hits, 1):
        src = h["metadata"]["source"]
        blocks.append(f"[{i}] (منبع: {src})\n{h['text']}")
    return "\n\n".join(blocks)

def generate_answer(query, hits, max_tokens=700):
    context = build_context(hits)
    user_msg = f"زمینه:\n{context}\n\nسوال: {query}"
    resp = client.chat.completions.create(
        model=GAPGPT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        max_tokens=max_tokens,
        temperature=0.2,
    )
    return resp.choices[0].message.content

# پایپ‌لاین کامل RAG: retrieve → rerank → generate
def rag_answer(query, pool=10, top_k=3):
    hits = retrieve_and_rerank(query, pool=pool, top_k=top_k)
    answer = generate_answer(query, hits)
    return answer, hits

# تست ۱: سوال داخل زمینه
ans, hits = rag_answer("علائم دیابت نوع ۲ چیست؟")
print("=== پاسخ ===\n", ans)
print("\n=== منابع استفاده‌شده ===")
for i, h in enumerate(hits, 1):
    print(f"[{i}] {h['metadata']['source']}")

**Out-of-Scope Testing**  
آزمایش رفتار سیستم در برابر سوالات خارج از حوزه دیابت

In [ ]:
# تست ۲: سوال خارج از دامنه (پاسخش در منابع دیابت نیست)
out_of_scope = "پایتخت فرانسه کجاست؟"
ans, hits = rag_answer(out_of_scope)
print("=== سوال خارج از زمینه ===")
print("سوال:", out_of_scope)
print("\nپاسخ:\n", ans)

# تست ۳: سوال پزشکی ولی بی‌ربط به دیابت
out_of_scope2 = "علائم سرطان ریه چیست؟"
ans2, hits2 = rag_answer(out_of_scope2)
print("\n=== سوال پزشکی نامرتبط ===")
print("سوال:", out_of_scope2)
print("\nپاسخ:\n", ans2)

**LLM Response Cache**  
کش‌سازی پاسخ‌های مدل زبانی با هش SHA-256 برای صرفه‌جویی در هزینه

In [ ]:
import json, hashlib, os

CACHE_PATH = "cache/llm_cache.json"
os.makedirs("cache", exist_ok=True)

def _load_cache():
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {}

def _save_cache(cache):
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

_llm_cache = _load_cache()

def cached_chat(messages, model=None, max_tokens=700, temperature=0.2):
    model = model or GAPGPT_MODEL
    # ساخت کلید یکتا از همه پارامترهای مؤثر بر پاسخ
    key_raw = json.dumps({"m": model, "msg": messages, "mt": max_tokens, "t": temperature},
                         ensure_ascii=False, sort_keys=True)
    key = hashlib.sha256(key_raw.encode("utf-8")).hexdigest()
    if key in _llm_cache:
        return _llm_cache[key]          # بدون پرداخت، از کش
    resp = client.chat.completions.create(
        model=model, messages=messages, max_tokens=max_tokens, temperature=temperature)
    answer = resp.choices[0].message.content
    _llm_cache[key] = answer
    _save_cache(_llm_cache)
    return answer

# تست: دو بار فراخوانی یکسان — بار دوم باید از کش بیاید
import time
msgs = [{"role": "user", "content": "بگو سلام"}]
t1 = time.time(); a1 = cached_chat(msgs, max_tokens=10); d1 = time.time()-t1
t2 = time.time(); a2 = cached_chat(msgs, max_tokens=10); d2 = time.time()-t2
print(f"بار اول : {d1:.2f}s | {a1}")
print(f"بار دوم : {d2:.3f}s | {a2}  (باید بسیار سریع‌تر باشد)")
print("اندازه کش:", len(_llm_cache), "ورودی")

**Evaluation Setup**  
تعریف ۱۵ سوال ارزیابی ثابت و نسخه کش‌دار تابع پاسخ‌ساز

In [ ]:
# نسخه کش‌دار generate_answer (جایگزین نسخه قبلی)
def generate_answer(query, hits, max_tokens=700):
    context = build_context(hits)
    user_msg = f"زمینه:\n{context}\n\nسوال: {query}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    return cached_chat(messages, max_tokens=max_tokens, temperature=0.2)

# ۱۵ سوال ارزیابی ثابت — در تمام فازهای بعد استفاده می‌شوند
EVAL_QUESTIONS = [
    "دیابت نوع ۲ چیست؟",
    "علائم شایع دیابت نوع ۲ کدامند؟",
    "تفاوت دیابت نوع ۱ و نوع ۲ چیست؟",
    "عوارض طولانی‌مدت دیابت نوع ۲ چیست؟",
    "چه کسانی بیشتر در معرض دیابت نوع ۲ هستند؟",
    "دیابت نوع ۲ چگونه تشخیص داده می‌شود؟",
    "نقش انسولین در بدن چیست؟",
    "مقاومت به انسولین یعنی چه؟",
    "آیا دیابت نوع ۲ قابل پیشگیری است؟",
    "سبک زندگی چه تأثیری بر دیابت نوع ۲ دارد؟",
    "چرا قند خون بالا برای بدن خطرناک است؟",
    "دیابت نوع ۲ چه ارتباطی با چاقی دارد؟",
    "آزمایش A1C چه چیزی را اندازه می‌گیرد؟",
    "آیا کودکان هم به دیابت نوع ۲ مبتلا می‌شوند؟",
    "مدیریت دیابت نوع ۲ شامل چه مواردی است؟",
]
print(f"تعداد سوالات ارزیابی: {len(EVAL_QUESTIONS)}")

# تست دوباره با نسخه کش‌دار
ans, _ = rag_answer(EVAL_QUESTIONS[1])
print("\nنمونه پاسخ کش‌دار:\n", ans[:300])

**Retrieval Evaluation — Recall & MRR**  
ارزیابی کیفیت بازیابی با متریک‌های Recall@k و Mean Reciprocal Rank

In [ ]:
# حقیقت پایه: برای هر سوال، کلیدواژه‌هایی که یک chunk مرتبط باید داشته باشد
GROUND_TRUTH = {
    "دیابت نوع ۲ چیست؟":                  ["مقاومت", "انسولین", "قند خون"],
    "علائم شایع دیابت نوع ۲ کدامند؟":      ["تشنگی", "ادرار", "خستگی"],
    "تفاوت دیابت نوع ۱ و نوع ۲ چیست؟":     ["نوع ۱", "انسولین", "تولید"],
}

def is_relevant(chunk_text, keywords):
    norm = normalize_fa(chunk_text)
    matched = sum(1 for kw in keywords if normalize_fa(kw) in norm)
    return matched >= 2  # حداقل ۲ کلیدواژه => مرتبط

def eval_retrieval(questions_gt, k=5, retriever=None):
    retriever = retriever or (lambda q: hybrid_search(q, k=k))
    recalls, rr = [], []
    for q, kws in questions_gt.items():
        hits = retriever(q)[:k]
        ranks = [i+1 for i, h in enumerate(hits) if is_relevant(h["text"], kws)]
        recalls.append(1.0 if ranks else 0.0)
        rr.append(1.0/ranks[0] if ranks else 0.0)
    import numpy as np
    return {"Recall@k": round(np.mean(recalls), 3), "MRR": round(np.mean(rr), 3)}

print("=== بازیابی Hybrid ===", eval_retrieval(GROUND_TRUTH, k=5))
print("=== بازیابی Dense  ===", eval_retrieval(GROUND_TRUTH, k=5,
      retriever=lambda q: dense_search(q, k=5)))
print("=== Hybrid + Rerank ===", eval_retrieval(GROUND_TRUTH, k=5,
      retriever=lambda q: retrieve_and_rerank(q, pool=10, top_k=5)))

**Generation Evaluation — LLM-as-Judge**  
ارزیابی کیفیت پاسخ مولد با امتیاز faithfulness و relevance توسط قضاوت مدل زبانی

In [ ]:
import json

JUDGE_PROMPT = """تو یک ارزیاب بی‌طرف هستی. بر اساس «زمینه»، «سوال» و «پاسخ»، دو امتیاز بده:
- faithfulness: آیا پاسخ فقط از زمینه استخراج شده و چیزی جعل نشده؟ (۱ تا ۵)
- relevance: آیا پاسخ مستقیماً به سوال جواب می‌دهد؟ (۱ تا ۵)
فقط یک JSON برگردان، بدون توضیح اضافه:
{"faithfulness": <عدد>, "relevance": <عدد>}"""

def judge_answer(query, answer, hits):
    context = build_context(hits)
    user_msg = f"زمینه:\n{context}\n\nسوال: {query}\n\nپاسخ:\n{answer}"
    raw = cached_chat(
        [{"role": "system", "content": JUDGE_PROMPT},
         {"role": "user", "content": user_msg}],
        max_tokens=60, temperature=0)
    try:
        clean = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(clean)
    except Exception:
        return {"faithfulness": None, "relevance": None, "raw": raw}

def eval_generation(questions):
    rows = []
    for q in questions:
        answer, hits = rag_answer(q)
        scores = judge_answer(q, answer, hits)
        rows.append({"سوال": q[:30], **scores})
    return rows

import numpy as np
dev_questions = EVAL_QUESTIONS[:3]
results = eval_generation(dev_questions)
for r in results:
    print(f"{r['سوال']:32} | faith={r.get('faithfulness')} | rel={r.get('relevance')}")

valid = [r for r in results if isinstance(r.get("faithfulness"), (int, float))]
if valid:
    print(f"\nمیانگین faithfulness: {np.mean([r['faithfulness'] for r in valid]):.2f}")
    print(f"میانگین relevance   : {np.mean([r['relevance'] for r in valid]):.2f}")

## جمع‌بندی فاز ۹ — ارزیابی

- **متریک‌های بازیابی** (بدون هزینه): Recall@k و MRR روی سه پیکربندی (Dense / Hybrid / Hybrid+Rerank).
- **متریک‌های مولد** (LLM-as-Judge، کش‌شده): faithfulness و answer-relevance.
- به‌صورت پیش‌فرض روی ۳ سوال توسعه اجرا می‌شود؛ برای اجرای کامل ۱۵ سوالی `RUN_FULL_EVAL = True` را ست کنید.

**Full Evaluation Pipeline**  
اجرای ارزیابی کامل روی همه سوالات و گزارش نتایج بازیابی و مولد

In [ ]:
RUN_FULL_EVAL = False   # برای اجرای کامل ۱۵ سوالی True کن

def full_evaluation(questions):
    # بازیابی: ground truth برای سوالاتی که تعریف شده
    gt_subset = {q: GROUND_TRUTH[q] for q in questions if q in GROUND_TRUTH}
    retrieval = {}
    if gt_subset:
        retrieval = {
            "Dense":          eval_retrieval(gt_subset, k=5, retriever=lambda q: dense_search(q, k=5)),
            "Hybrid":         eval_retrieval(gt_subset, k=5),
            "Hybrid+Rerank":  eval_retrieval(gt_subset, k=5, retriever=lambda q: retrieve_and_rerank(q, pool=10, top_k=5)),
        }
    # مولد
    gen_rows = eval_generation(questions)
    valid = [r for r in gen_rows if isinstance(r.get("faithfulness"), (int, float))]
    gen_summary = {}
    if valid:
        gen_summary = {
            "faithfulness": round(np.mean([r["faithfulness"] for r in valid]), 2),
            "relevance":    round(np.mean([r["relevance"] for r in valid]), 2),
        }
    return retrieval, gen_summary

questions = EVAL_QUESTIONS if RUN_FULL_EVAL else EVAL_QUESTIONS[:3]
print(f"ارزیابی روی {len(questions)} سوال...\n")
retrieval, gen_summary = full_evaluation(questions)

print("=== متریک‌های بازیابی ===")
for name, m in retrieval.items():
    print(f"{name:16} | {m}")
print("\n=== متریک‌های مولد (میانگین) ===")
print(gen_summary)

**Gradio UI**  
رابط کاربری تعاملی فارسی با Gradio و فونت وزیر برای استفاده نهایی

In [ ]:
import os, base64, glob
import gradio as gr

# پیدا کردن فایل فونت وزیر در assets (هر نامی: Vazir / Vazirmatn)
font_candidates = glob.glob("assets/fonts/*Vazir*.ttf") + glob.glob("assets/fonts/*vazir*.ttf")
assert font_candidates, "فایل فونت وزیر در assets/ پیدا نشد!"
FONT_PATH = font_candidates[0]
print("فونت:", FONT_PATH)

# تبدیل فونت به base64 برای جاسازی در CSS
with open(FONT_PATH, "rb") as f:
    font_b64 = base64.b64encode(f.read()).decode()

def rag_for_ui(query, source_filter):
    if not query.strip():
        return "لطفاً سوال خود را وارد کنید.", ""
    if source_filter == "همه":
        hits = retrieve_and_rerank(query, pool=10, top_k=3)
    else:
        candidates = hybrid_search(query, k=10)
        candidates = [c for c in candidates if c["metadata"]["source_type"] == source_filter]
        hits = rerank(query, candidates, top_k=3) if candidates else []
    if not hits:
        return "بر اساس منابع موجود، پاسخی یافت نشد.", ""
    answer = generate_answer(query, hits)
    sources = "\n".join(
        f"[{i}] {h['metadata']['source']} (نوع: {h['metadata']['source_type']})"
        for i, h in enumerate(hits, 1))
    return answer, sources    

custom_css = f"""
@font-face {{
    font-family: 'Vazir';
    src: url(data:font/ttf;base64,{font_b64}) format('truetype');
}}
.gradio-container, .gradio-container * {{
    font-family: 'Vazir', sans-serif !important;
    direction: rtl !important;
    text-align: right !important;
}}
textarea, input {{ direction: rtl !important; text-align: right !important; }}
"""

with gr.Blocks(title="دستیار دیابت نوع ۲", css=custom_css) as demo:
    gr.Markdown("## 🩺 دستیار پاسخ‌گوی دیابت نوع ۲ (RAG)")
    gr.Markdown("سوال خود را درباره دیابت نوع ۲ بپرسید. پاسخ فقط بر اساس منابع موجود داده می‌شود.")
    with gr.Row():
        q_in = gr.Textbox(label="سوال شما", placeholder="مثلاً: علائم دیابت نوع ۲ چیست؟", scale=4)
        filt = gr.Dropdown(["همه", "pdf", "web", "faq"], value="همه", label="فیلتر منبع", scale=1)
    btn = gr.Button("پرسش", variant="primary")
    ans_out = gr.Textbox(label="پاسخ", lines=6)
    src_out = gr.Textbox(label="منابع استفاده‌شده", lines=3)
    btn.click(rag_for_ui, inputs=[q_in, filt], outputs=[ans_out, src_out])
    gr.Examples(EVAL_QUESTIONS[:5], inputs=q_in)

demo.launch(inbrowser=True)

**Query Rewriting**  
تبدیل سوال مبهم پیرو به سوال مستقل با استفاده از تاریخچه گفتگو

In [ ]:
REWRITE_PROMPT = """با توجه به «تاریخچه گفتگو» و «سوال جدید»، سوال جدید را به یک سوال
کامل و مستقل بازنویسی کن که بدون نیاز به تاریخچه قابل‌فهم باشد.
ضمیرها و ارجاعات مبهم (مثل «آن»، «درمانش») را با موضوع واقعی جایگزین کن.
فقط سوال بازنویسی‌شده را برگردان، بدون توضیح اضافه.
اگر سوال از قبل مستقل است، همان را بدون تغییر برگردان."""

def rewrite_query(query, history):
    if not history:
        return query
    hist_text = "\n".join(f"کاربر: {h['q']}\nدستیار: {h['a'][:100]}" for h in history[-3:])
    user_msg = f"تاریخچه گفتگو:\n{hist_text}\n\nسوال جدید: {query}"
    rewritten = cached_chat(
        [{"role": "system", "content": REWRITE_PROMPT},
         {"role": "user", "content": user_msg}],
        max_tokens=80, temperature=0)
    return rewritten.strip()

# تست: شبیه‌سازی یک گفتگوی دو مرحله‌ای
history = [{"q": "علائم دیابت نوع ۲ چیست؟", "a": "افزایش تشنگی، تکرر ادرار و خستگی..."}]
followup = "و درمانش چطور؟"
rewritten = rewrite_query(followup, history)
print("سوال پیرو :", followup)
print("بازنویسی  :", rewritten)

**Conversational RAG**   
نگهداری تاریخچه، بازنویسی خودکار سوال پیرو، و پاسخ مبتنی بر زمینه

In [ ]:
class ConversationalRAG:
    def __init__(self):
        self.history = []

    def ask(self, query, top_k=3, verbose=True):
        # ۱) بازنویسی سوال با کمک تاریخچه
        standalone = rewrite_query(query, self.history)
        # ۲) بازیابی + بازرتبه‌بندی با کوئری مستقل
        hits = retrieve_and_rerank(standalone, pool=10, top_k=top_k)
        # ۳) تولید پاسخ
        answer = generate_answer(standalone, hits) if hits else "پاسخی یافت نشد."
        # ۴) افزودن به حافظه
        self.history.append({"q": query, "a": answer})
        if verbose:
            if standalone != query:
                print(f"↻ بازنویسی: {standalone}")
            print(f"پاسخ: {answer}\n")
        return answer

    def reset(self):
        self.history = []

# تست: یک گفتگوی سه‌مرحله‌ای
chat = ConversationalRAG()
print("Q1:", "علائم دیابت نوع ۲ چیست؟")
chat.ask("علائم دیابت نوع ۲ چیست؟")
print("Q2:", "و عوارضش؟")
chat.ask("و عوارضش؟")
print("Q3:", "چطور پیشگیری کنم؟")
chat.ask("چطور پیشگیری کنم؟")
print("تعداد پیام در حافظه:", len(chat.history))

**HyDE — Hypothetical Document Embeddings**   
ساخت پاسخ فرضی برای کوئری و جستجو با امبدینگ آن به‌جای سوال خام

In [ ]:
HYDE_PROMPT = """به سوال زیر یک پاسخ کوتاه و علمی (۲ تا ۳ جمله) به فارسی بده،
طوری که شبیه متن یک منبع آموزشی پزشکی باشد. فقط پاسخ را بنویس."""

def hyde_search(query, k=5, pool=10):
    # ۱) ساخت پاسخ فرضی (کش‌شده)
    hypo = cached_chat(
        [{"role": "system", "content": HYDE_PROMPT},
         {"role": "user", "content": query}],
        max_tokens=120, temperature=0.3).strip()
    # ۲) جستجوی dense با امبدینگ پاسخ فرضی به‌جای سوال
    q_emb = embedder.encode(f"query: {hypo}", normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=k)
    hits = [{"text": d, "metadata": m, "score": 1 - dist}
            for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]
    return hits, hypo

# تست: مقایسه بازیابی معمولی با HyDE
query = "چرا قند خون بالا خطرناک است؟"
print("=== Dense معمولی ===")
for h in dense_search(query, k=3):
    print(f"  {h['metadata']['source']} | {h['text'][:60]}")

print("\n=== HyDE ===")
hyde_hits, hypo = hyde_search(query, k=3)
print(f"پاسخ فرضی: {hypo[:120]}...\n")
for h in hyde_hits:
    print(f"  {h['metadata']['source']} | {h['text'][:60]}")

**Multimodal Indexing**   
نگاشت متن و تصویر به فضای مشترک و ساخت بردار برای تصاویر assets/images

In [ ]:
import glob
from PIL import Image
from sentence_transformers import SentenceTransformer

# لود مدل CLIP لوکال (آفلاین)
clip_model = SentenceTransformer("models/clip-ViT-B-32", device="cpu")

# جمع‌آوری مسیر تصاویر (هر دو فرمت jpg و png)
image_paths = sorted(glob.glob("assets/images/*.jpg") + glob.glob("assets/images/*.png"))
assert image_paths, "تصویری در assets/images/ پیدا نشد!"

# ساخت embedding برای هر تصویر
images = [Image.open(p).convert("RGB") for p in image_paths]
image_embeddings = clip_model.encode(images, normalize_embeddings=True)

print(f"تعداد تصاویر ایندکس‌شده: {len(image_paths)}")
for p in image_paths:
    print("  -", os.path.basename(p))
print("بُعد بردار CLIP:", image_embeddings.shape[1])

**Text-to-Image Search**   
یافتن مرتبط‌ترین تصویر برای یک کوئری متنی با شباهت CLIP

In [ ]:
import numpy as np

def translate_for_clip(query):
    # ترجمه کوتاه فارسی→انگلیسی برای تطبیق بهتر CLIP (کش‌شده)
    return cached_chat(
        [{"role": "system", "content": "این عبارت فارسی را به یک عبارت کوتاه انگلیسی ترجمه کن. فقط ترجمه را بنویس."},
         {"role": "user", "content": query}],
        max_tokens=30, temperature=0).strip()

def image_search(query, top_k=3, translate=True):
    text = translate_for_clip(query) if translate else query
    q_emb = clip_model.encode(text, normalize_embeddings=True)
    sims = np.dot(image_embeddings, q_emb)  # شباهت کسینوسی (بردارها نرمال‌اند)
    top = np.argsort(sims)[::-1][:top_k]
    return [{"path": image_paths[i],
             "name": os.path.basename(image_paths[i]),
             "score": float(sims[i])} for i in top]

# تست
query = "نمودار قند خون"
print(f"کوئری: {query}")
print(f"ترجمه CLIP: {translate_for_clip(query)}\n")
for i, r in enumerate(image_search(query, top_k=3), 1):
    print(f"#{i} | score={r['score']:.3f} | {r['name']}")

**Visual Result**  
رندر مرتبط‌ترین تصویر یافته‌شده برای کوئری در خروجی نوت‌بوک

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def show_image_results(query, top_k=3):
    results = image_search(query, top_k=top_k)
    fig, axes = plt.subplots(1, len(results), figsize=(4*len(results), 4))
    if len(results) == 1:
        axes = [axes]
    for ax, r in zip(axes, results):
        ax.imshow(Image.open(r["path"]))
        ax.set_title(f"{r['score']:.3f}", fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return results

# تست بصری
show_image_results("نمودار تشخیص دیابت", top_k=3);

**Multimodal Gradio UI**   
رابط راست‌چین با فونت وزیر که پاسخ متنی، منابع و تصویر مرتبط را نمایش می‌دهد

In [ ]:
import glob, base64
import gradio as gr

# پیدا کردن فونت وزیر در مسیر جدید
_fonts = glob.glob("assets/fonts/*Vazir*.ttf") + glob.glob("assets/fonts/*vazir*.ttf")
font_b64 = base64.b64encode(open(_fonts[0], "rb").read()).decode() if _fonts else ""
custom_css = f"""
@font-face {{ font-family:'Vazir'; src:url(data:font/ttf;base64,{font_b64}) format('truetype'); }}
.gradio-container, .gradio-container * {{ font-family:'Vazir',sans-serif !important;
  direction:rtl !important; text-align:right !important; }}
""" if font_b64 else ""

def rag_multimodal(query, source_filter):
    if not query.strip():
        return "لطفاً سوال خود را وارد کنید.", "", None
    if source_filter == "همه":
        hits = retrieve_and_rerank(query, pool=10, top_k=3)
    else:
        cands = [c for c in hybrid_search(query, k=10) if c["metadata"]["source_type"] == source_filter]
        hits = rerank(query, cands, top_k=3) if cands else []
    if not hits:
        return "بر اساس منابع موجود، پاسخی یافت نشد.", "", None
    answer = generate_answer(query, hits)
    sources = "\n".join(f"[{i}] {h['metadata']['source']} ({h['metadata']['source_type']})"
                        for i, h in enumerate(hits, 1))
    img = image_search(query, top_k=1)[0]["path"]   # مرتبط‌ترین تصویر
    return answer, sources, img

with gr.Blocks(title="دستیار دیابت نوع ۲", css=custom_css) as demo:
    gr.Markdown("## 🩺 دستیار چندوجهی دیابت نوع ۲ (RAG)")
    with gr.Row():
        q_in = gr.Textbox(label="سوال شما", scale=4)
        filt = gr.Dropdown(["همه","pdf","web","faq"], value="همه", label="فیلتر منبع", scale=1)
    btn = gr.Button("پرسش", variant="primary")
    with gr.Row():
        with gr.Column(scale=2):
            ans_out = gr.Textbox(label="پاسخ", lines=6)
            src_out = gr.Textbox(label="منابع", lines=3)
        img_out = gr.Image(label="تصویر مرتبط", scale=1)
    btn.click(rag_multimodal, [q_in, filt], [ans_out, src_out, img_out])
    gr.Examples(EVAL_QUESTIONS[:5], inputs=q_in)

demo.launch(inbrowser=True)